# Text Embedding with vLLM + Qwen3-Embedding-8B

This notebook generates embeddings for user and subreddit text data using:
- **Model**: Qwen/Qwen3-Embedding-8B
- **Framework**: vLLM (for fast inference)
- **Output dimensions**: 4096 (full) and 128 (MRL/Matryoshka)

**Memory-efficient version**: Embeddings are saved incrementally to disk and consolidated at the end.

Make sure to set runtime to **GPU** (T4 or better, A100 recommended for 8B model)

## 1. Setup and Installation

In [ ]:
# # Check GPU availability first
# import torch
# print(f"CUDA available: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")
#     print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
# else:
#     raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > GPU")

In [ ]:
import os
import time

print("🚀 Installing vLLM and forcing dependency upgrades...")

# 1. Force install vLLM and upgrade the conflicting libraries
# We use --extra-index-url to ensure we get the right CUDA wheels
!pip install vllm>=0.8.5 \
    fastapi uvicorn \
    --upgrade \
    --force-reinstall \
    --no-cache-dir

print("\n✅ Installation complete.")
print("🔄 RESTARTING RUNTIME AUTOMATICALLY NOW...")
print("   (Please wait 10 seconds, then run the next cell)")

# 2. Kill the runtime to force a reload of the new libraries
os.kill(os.getpid(), 9)

In [ ]:
try:
    import vllm
    print(f"✅ SUCCESS: vLLM version {vllm.__version__} is installed and importable!")
except ImportError as e:
    print(f"❌ FAILURE: Still cannot import vLLM. Error: {e}")
except Exception as e:
    print(f"⚠️ Unexpected error: {e}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import time
import shutil
import glob

## 2. Configuration

In [ ]:
import os

# ============================================================
# CONFIGURATION - Edit these paths as needed
# ============================================================

# Input directories containing the JSON files
# UPDATED: Pointing to the new Chi-Square (_chi2) output folders
INPUT_DIRS = {
    "v4_chi2": "/content/drive/MyDrive/CIS 5300/engage_corpus_processed_v4_chi2/text_context_filtered",
    "v3_5_chi2": "/content/drive/MyDrive/CIS 5300/engage_corpus_processed_v3_5_chi2/text_context_filtered",
}

# Output directory for embeddings
OUTPUT_BASE_DIR = "/content/drive/MyDrive/CIS 5300/embeddings_qwen3_8b"

# Temporary directory for intermediate saves (local for speed)
TEMP_DIR = "/content/temp_embeddings"

# Model settings
MODEL_NAME = "Qwen/Qwen3-Embedding-8B"
FULL_DIM = 4096  # Full embedding dimension
MRL_DIM = 128    # Matryoshka reduced dimension

# Processing settings
BATCH_SIZE = 2048   # Reduce if OOM errors (8B model is large)
SAVE_EVERY_N_BATCHES = 10  # Save intermediate results every N batches

# Create temp directory
os.makedirs(TEMP_DIR, exist_ok=True)

# Verify input directories exist
for name, path in INPUT_DIRS.items():
    if os.path.exists(path):
        print(f"✓ Found {name}: {path}")
        print(f"  Files: {os.listdir(path)}")
    else:
        print(f"✗ NOT FOUND {name}: {path}")

In [ ]:
import json
import os

# Paths to the source data
# UPDATED: Pointing to the new Chi-Square (_chi2) output folders
INPUT_DIRS = {
    "v4_chi2": "/content/drive/MyDrive/CIS 5300/engage_corpus_processed_v4_chi2/text_context_filtered",
    "v3_5_chi2": "/content/drive/MyDrive/CIS 5300/engage_corpus_processed_v3_5_chi2/text_context_filtered",
}

splits = ['train', 'dev', 'test']

print("="*70)
print("SUBREDDIT COUNTS IN SOURCE DATA")
print("="*70)

for dataset_name, input_dir in INPUT_DIRS.items():
    print(f"\n{dataset_name}:")
    print("-"*50)

    if not os.path.exists(input_dir):
        print(f"  ✗ Directory not found: {input_dir}")
        continue

    all_subreddits = set()

    for split in splits:
        subreddit_file = os.path.join(input_dir, f'subreddit_text_{split}.json')

        if os.path.exists(subreddit_file):
            with open(subreddit_file, 'r') as f:
                subreddit_text_dict = json.load(f)

            num_subreddits = len(subreddit_text_dict)
            all_subreddits.update(subreddit_text_dict.keys())

            print(f"  {split:5s}: {num_subreddits:,} subreddits")
        else:
            print(f"  {split:5s}: ✗ File not found")

    print(f"\n  Total unique subreddits across all splits: {len(all_subreddits):,}")

# Optional: Compare the two datasets
print("\n" + "="*70)
print("COMPARISON")
print("="*70)

# Load all subreddits from both datasets
v4_subreddits = set()
v3_5_subreddits = set()

for split in splits:
    # v4_chi2
    v4_file = os.path.join(INPUT_DIRS["v4_chi2"], f'subreddit_text_{split}.json')
    if os.path.exists(v4_file):
        with open(v4_file, 'r') as f:
            v4_subreddits.update(json.load(f).keys())

    # v3_5_chi2
    v3_5_file = os.path.join(INPUT_DIRS["v3_5_chi2"], f'subreddit_text_{split}.json')
    if os.path.exists(v3_5_file):
        with open(v3_5_file, 'r') as f:
            v3_5_subreddits.update(json.load(f).keys())

print(f"\nSubreddits in v4_chi2 but NOT in v3_5_chi2: {len(v4_subreddits - v3_5_subreddits):,}")
print(f"Subreddits in v3_5_chi2 but NOT in v4_chi2: {len(v3_5_subreddits - v4_subreddits):,}")
print(f"Subreddits in BOTH datasets: {len(v4_subreddits & v3_5_subreddits):,}")

# Optional: Show a few examples of subreddits that differ
only_in_v4 = v4_subreddits - v3_5_subreddits
only_in_v3_5 = v3_5_subreddits - v4_subreddits

if only_in_v4:
    print(f"\nExample subreddits only in v4 (first 10):")
    print(f"  {sorted(only_in_v4)[:10]}")

if only_in_v3_5:
    print(f"\nExample subreddits only in v3.5 (first 10):")
    print(f"  {sorted(only_in_v3_5)[:10]}")

In [ ]:
import json
import numpy as np
import os

# Path to embeddings output
# UPDATED: Checking the NEW v3.5 Chi-Square embeddings folder
EMBEDDINGS_DIR = "/content/drive/MyDrive/CIS 5300/embeddings_qwen3_8b/v3_5_chi2/filtered"

splits = ['train', 'dev', 'test']

print("="*70)
print("SUBREDDIT COUNTS IN SAVED EMBEDDINGS (v3_5_chi2)")
print("="*70)

all_subreddits_in_embeddings = set()

for split in splits:
    split_dir = os.path.join(EMBEDDINGS_DIR, split)
    subreddit_names_file = os.path.join(split_dir, 'subreddit_names.json')

    print(f"\n{split.upper()}:")
    print("-"*50)

    if os.path.exists(subreddit_names_file):
        # Load subreddit names
        with open(subreddit_names_file, 'r') as f:
            subreddit_names = json.load(f)

        num_subreddits = len(subreddit_names)
        all_subreddits_in_embeddings.update(subreddit_names)

        print(f"  Subreddits in subreddit_names.json: {num_subreddits:,}")

        # Verify against embedding array shape
        emb_128_file = os.path.join(split_dir, 'subreddit_embeddings_128.npy')
        emb_4096_file = os.path.join(split_dir, 'subreddit_embeddings_4096.npy')

        if os.path.exists(emb_128_file):
            emb_128 = np.load(emb_128_file)
            print(f"  subreddit_embeddings_128.npy shape: {emb_128.shape}")
            print(f"    → {emb_128.shape[0]:,} rows (embeddings)")

        if os.path.exists(emb_4096_file):
            emb_4096 = np.load(emb_4096_file)
            print(f"  subreddit_embeddings_4096.npy shape: {emb_4096.shape}")
            print(f"    → {emb_4096.shape[0]:,} rows (embeddings)")

        # Verify consistency
        if os.path.exists(emb_128_file) and os.path.exists(emb_4096_file):
            if emb_128.shape[0] == emb_4096.shape[0] == num_subreddits:
                print(f"  ✓ Counts match: {num_subreddits:,} subreddits")
            else:
                print(f"  ✗ MISMATCH!")
                print(f"    JSON: {num_subreddits:,}, 128-dim: {emb_128.shape[0]:,}, 4096-dim: {emb_4096.shape[0]:,}")

        # Show first few subreddit names
        print(f"\n  First 10 subreddits:")
        for i, name in enumerate(subreddit_names[:10]):
            print(f"    [{i}] {name}")

    else:
        print(f"  ✗ File not found: {subreddit_names_file}")

print(f"\n{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"Total unique subreddits across all splits: {len(all_subreddits_in_embeddings):,}")

In [ ]:
import json
import numpy as np
import os

# Path to embeddings output
# UPDATED: Checking the NEW v3.5 Chi-Square embeddings folder
EMBEDDINGS_DIR = "/content/drive/MyDrive/CIS 5300/embeddings_qwen3_8b/v3_5_chi2/filtered"

splits = ['train', 'dev', 'test']

print("="*70)
print("FILE STRUCTURE AND ATTRIBUTES (v3_5_chi2)")
print("="*70)

for split in splits:
    split_dir = os.path.join(EMBEDDINGS_DIR, split)

    print(f"\n{'='*70}")
    print(f"{split.upper()} SPLIT")
    print(f"{'='*70}")

    if not os.path.exists(split_dir):
        print(f"  ✗ Directory not found: {split_dir}")
        continue

    # List all files in the directory
    files = sorted(os.listdir(split_dir))

    for filename in files:
        filepath = os.path.join(split_dir, filename)
        filesize_mb = os.path.getsize(filepath) / (1024 * 1024)

        print(f"\n{filename}")
        print(f"  File size: {filesize_mb:.2f} MB")

        # Handle JSON files
        if filename.endswith('.json'):
            with open(filepath, 'r') as f:
                data = json.load(f)

            print(f"  Type: list")
            print(f"  Length: {len(data):,}")
            print(f"  Data type of elements: {type(data[0]).__name__ if data else 'N/A'}")
            print(f"  First 5 elements: {data[:5]}")
            print(f"  Last 5 elements: {data[-5:]}")

            # Check for duplicates
            if isinstance(data, list):
                unique_count = len(set(data))
                if unique_count != len(data):
                    print(f"  ⚠ WARNING: {len(data) - unique_count} duplicate entries!")
                else:
                    print(f"  ✓ All entries are unique")

        # Handle NumPy files
        elif filename.endswith('.npy'):
            arr = np.load(filepath)

            print(f"  Type: numpy.ndarray")
            print(f"  Shape: {arr.shape}")
            print(f"  Dtype: {arr.dtype}")
            print(f"  Number of dimensions: {arr.ndim}")
            print(f"  Total elements: {arr.size:,}")
            print(f"  Memory size: {arr.nbytes / (1024*1024):.2f} MB")

            # Statistical info
            print(f"\n  Statistics:")
            print(f"    Min value: {arr.min():.6f}")
            print(f"    Max value: {arr.max():.6f}")
            print(f"    Mean: {arr.mean():.6f}")
            print(f"    Std: {arr.std():.6f}")

            # Check for NaN values
            nan_count = np.isnan(arr).sum()
            if nan_count > 0:
                print(f"    ⚠ NaN values: {nan_count:,} ({100*nan_count/arr.size:.2f}%)")
            else:
                print(f"    ✓ No NaN values")

            # For embeddings, check L2 normalization
            if arr.ndim == 2:
                norms = np.linalg.norm(arr, axis=1)
                print(f"\n  L2 Norms:")
                print(f"    Min norm: {norms.min():.6f}")
                print(f"    Max norm: {norms.max():.6f}")
                print(f"    Mean norm: {norms.mean():.6f}")
                if np.allclose(norms, 1.0, atol=1e-5):
                    print(f"    ✓ Embeddings are L2-normalized")
                else:
                    print(f"    ⚠ Embeddings may not be normalized")

                # Show first row sample
                print(f"\n  First embedding (first 10 dims): {arr[0, :10]}")

print("\n" + "="*70)
print("INSPECTION COMPLETE")
print("="*70)

## 3. Load the vLLM Model



In [ ]:
from vllm import LLM, PoolingParams

print(f"Loading {MODEL_NAME}...")
print("This may take several minutes for the 8B model...")
start_time = time.time()

# Load model with Matryoshka support
model = LLM(
    model=MODEL_NAME,
    task="embed",
    trust_remote_code=True,
    dtype="half",  # Use fp16 to save memory
    gpu_memory_utilization=0.9,  # Use most of GPU memory
    max_model_len=32000,  # Max sequence length
    hf_overrides={
        "is_matryoshka": True,
        "matryoshka_dimensions": [32, 64, 128, 256, 512, 768, 1024, 2048, 4096]
    },
)

print(f"\n✓ Model loaded in {time.time() - start_time:.1f}s")

## 4. Memory-Efficient Embedding Functions


In [ ]:
# def get_detailed_instruct(task_description: str, query: str) -> str:
#     """Format instruction for Qwen3 embedding model."""
#     return f'Instruct: {task_description}\nQuery: {query}'


def embed_and_save_incrementally(
    model,
    texts: list,
    task_description: str,
    temp_dir: str,
    prefix: str,
    dimension: int,
    batch_size: int = 8,
    save_every_n_batches: int = 10
) -> list:
    """
    Embed texts and save incrementally to disk.

    Args:
        model: vLLM LLM model
        texts: List of text strings
        task_description: Task description for instruction
        temp_dir: Directory for intermediate saves
        prefix: Filename prefix for intermediate files
        dimension: Output embedding dimension
        batch_size: Batch size for processing
        save_every_n_batches: Save to disk every N batches

    Returns:
        List of paths to intermediate .npy files
    """
    pooling_params = PoolingParams(dimensions=dimension)

    intermediate_files = []
    current_embeddings = []
    file_counter = 0
    batch_counter = 0

    total_batches = (len(texts) + batch_size - 1) // batch_size

    pbar = tqdm(range(0, len(texts), batch_size),
                desc=f"Embedding (dim={dimension})",
                total=total_batches)

    for i in pbar:
        batch_texts = texts[i:i+batch_size]

        # Add instruction to each text
        # batch_with_instruct = [
        #     get_detailed_instruct(task_description, text)
        #     for text in batch_texts
        # ]

        # Get embeddings
        outputs = model.embed(batch_texts, pooling_params=pooling_params)

        # Extract and normalize embeddings
        batch_embeddings = np.array([o.outputs.embedding for o in outputs], dtype=np.float32)
        norms = np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
        batch_embeddings = batch_embeddings / norms

        current_embeddings.append(batch_embeddings)
        batch_counter += 1

        # Save intermediate results periodically
        if batch_counter >= save_every_n_batches:
            # Concatenate and save
            chunk = np.vstack(current_embeddings)
            chunk_path = os.path.join(temp_dir, f"{prefix}_chunk_{file_counter:04d}.npy")
            np.save(chunk_path, chunk)
            intermediate_files.append(chunk_path)

            pbar.set_postfix({"saved_chunks": len(intermediate_files), "chunk_size": chunk.shape[0]})

            # Clear memory
            current_embeddings = []
            batch_counter = 0
            file_counter += 1

    # Save any remaining embeddings
    if current_embeddings:
        chunk = np.vstack(current_embeddings)
        chunk_path = os.path.join(temp_dir, f"{prefix}_chunk_{file_counter:04d}.npy")
        np.save(chunk_path, chunk)
        intermediate_files.append(chunk_path)

    return intermediate_files


def consolidate_embeddings(intermediate_files: list, output_path: str) -> np.ndarray:
    """
    Load intermediate files and consolidate into single array.

    Args:
        intermediate_files: List of paths to .npy chunk files
        output_path: Path to save consolidated embeddings

    Returns:
        Consolidated numpy array
    """
    print(f"  Consolidating {len(intermediate_files)} chunks...")

    # Load all chunks
    chunks = []
    for f in tqdm(intermediate_files, desc="  Loading chunks"):
        chunks.append(np.load(f))

    # Concatenate
    consolidated = np.vstack(chunks)

    # Save to final location
    np.save(output_path, consolidated)
    print(f"  ✓ Saved: {output_path} (shape: {consolidated.shape})")

    # Clean up intermediate files
    for f in intermediate_files:
        os.remove(f)

    return consolidated

In [ ]:
def process_user_text_incremental(
    input_path: str,
    output_dir: str,
    temp_dir: str,
    model,
    batch_size: int = 8,
    save_every_n_batches: int = 10
):
    """
    Process user text JSON file with incremental saves.
    """
    print(f"  Loading: {input_path}")
    with open(input_path, 'r') as f:
        user_text_dict = json.load(f)

    # Sort by user ID to maintain consistent order
    user_ids = sorted([int(k) for k in user_text_dict.keys()])
    user_texts = [user_text_dict[str(uid)] for uid in user_ids]

    print(f"  Found {len(user_texts)} users")

    # Save user IDs immediately
    with open(os.path.join(output_dir, 'user_ids.json'), 'w') as f:
        json.dump(user_ids, f)

    # Clear the dict from memory
    del user_text_dict

    task_desc = "Given user Reddit activity, retrieve relevant subreddit communities"

    # Generate and save FULL embeddings (4096-dim)
    print(f"\n  Generating {FULL_DIM}-dim embeddings (with incremental saves)...")
    full_files = embed_and_save_incrementally(
        model, user_texts, task_desc,
        temp_dir=temp_dir,
        prefix="user_full",
        dimension=FULL_DIM,
        batch_size=batch_size,
        save_every_n_batches=save_every_n_batches
    )

    # Consolidate full embeddings
    consolidate_embeddings(
        full_files,
        os.path.join(output_dir, f'user_embeddings_{FULL_DIM}.npy')
    )

    # Generate and save MRL embeddings (128-dim)
    print(f"\n  Generating {MRL_DIM}-dim MRL embeddings (with incremental saves)...")
    mrl_files = embed_and_save_incrementally(
        model, user_texts, task_desc,
        temp_dir=temp_dir,
        prefix="user_mrl",
        dimension=MRL_DIM,
        batch_size=batch_size,
        save_every_n_batches=save_every_n_batches
    )

    # Consolidate MRL embeddings
    consolidate_embeddings(
        mrl_files,
        os.path.join(output_dir, f'user_embeddings_{MRL_DIM}.npy')
    )

    # Clear texts from memory
    del user_texts
    print(f"  ✓ User embeddings complete!")


def process_subreddit_text_incremental(
    input_path: str,
    output_dir: str,
    temp_dir: str,
    model,
    batch_size: int = 8,
    save_every_n_batches: int = 10
):
    """
    Process subreddit text JSON file with incremental saves.
    """
    print(f"  Loading: {input_path}")
    with open(input_path, 'r') as f:
        subreddit_text_dict = json.load(f)

    # Get subreddit names and texts
    subreddit_names = list(subreddit_text_dict.keys())
    subreddit_texts = list(subreddit_text_dict.values())

    print(f"  Found {len(subreddit_texts)} subreddits")

    # Save subreddit names immediately
    with open(os.path.join(output_dir, 'subreddit_names.json'), 'w') as f:
        json.dump(subreddit_names, f)

    # Clear the dict from memory
    del subreddit_text_dict
    del subreddit_names

    task_desc = "Given subreddit community content, retrieve similar communities"

    # Generate and save FULL embeddings (4096-dim)
    print(f"\n  Generating {FULL_DIM}-dim embeddings (with incremental saves)...")
    full_files = embed_and_save_incrementally(
        model, subreddit_texts, task_desc,
        temp_dir=temp_dir,
        prefix="subreddit_full",
        dimension=FULL_DIM,
        batch_size=batch_size,
        save_every_n_batches=save_every_n_batches
    )

    # Consolidate full embeddings
    consolidate_embeddings(
        full_files,
        os.path.join(output_dir, f'subreddit_embeddings_{FULL_DIM}.npy')
    )

    # Generate and save MRL embeddings (128-dim)
    print(f"\n  Generating {MRL_DIM}-dim MRL embeddings (with incremental saves)...")
    mrl_files = embed_and_save_incrementally(
        model, subreddit_texts, task_desc,
        temp_dir=temp_dir,
        prefix="subreddit_mrl",
        dimension=MRL_DIM,
        batch_size=batch_size,
        save_every_n_batches=save_every_n_batches
    )

    # Consolidate MRL embeddings
    consolidate_embeddings(
        mrl_files,
        os.path.join(output_dir, f'subreddit_embeddings_{MRL_DIM}.npy')
    )

    # Clear texts from memory
    del subreddit_texts
    print(f"  ✓ Subreddit embeddings complete!")

## 5. Quick Test (Optional)

In [ ]:
# Quick test to make sure everything works
test_texts = [
    "Machine learning is fascinating",
    "I love playing video games",
    "Python programming is useful",
]

print("Testing incremental embedding...")

# Create test temp dir
test_temp = os.path.join(TEMP_DIR, "test")
os.makedirs(test_temp, exist_ok=True)

# Test with small save interval
test_files = embed_and_save_incrementally(
    model, test_texts, "Test task",
    temp_dir=test_temp,
    prefix="test",
    dimension=MRL_DIM,
    batch_size=2,
    save_every_n_batches=1
)

print(f"Created {len(test_files)} intermediate files")

# Consolidate
test_output = os.path.join(test_temp, "test_consolidated.npy")
test_emb = consolidate_embeddings(test_files, test_output)

print(f"Final shape: {test_emb.shape}")
print(f"L2 norms (should be ~1.0): {np.linalg.norm(test_emb, axis=1)}")

# Cleanup test
shutil.rmtree(test_temp)
print("\n✓ Test passed!")

## 6. Process All Data

In [ ]:
def process_dataset_incremental(
    dataset_name: str,
    input_dir: str,
    output_base: str,
    temp_base: str,
    model,
    batch_size: int = 8,
    save_every_n_batches: int = 10
):
    """
    Process a complete dataset with incremental saves.
    """
    print(f"\n{'='*70}")
    print(f"Processing dataset: {dataset_name}")
    print(f"Input: {input_dir}")
    print(f"{'='*70}")

    output_dir = os.path.join(output_base, dataset_name, "filtered")
    temp_dir = os.path.join(temp_base, dataset_name)

    splits = ['train', 'dev', 'test']

    for split in splits:
        print(f"\n{'─'*50}")
        print(f"Processing {split} split")
        print(f"{'─'*50}")

        split_output_dir = os.path.join(output_dir, split)
        split_temp_dir = os.path.join(temp_dir, split)

        os.makedirs(split_output_dir, exist_ok=True)
        os.makedirs(split_temp_dir, exist_ok=True)

        # Process user text
        user_file = os.path.join(input_dir, f'user_text_{split}.json')
        if os.path.exists(user_file):
            print(f"\n[USER TEXT]")
            process_user_text_incremental(
                input_path=user_file,
                output_dir=split_output_dir,
                temp_dir=split_temp_dir,
                model=model,
                batch_size=batch_size,
                save_every_n_batches=save_every_n_batches
            )
        else:
            print(f"  ✗ User file not found: {user_file}")

        # Process subreddit text
        subreddit_file = os.path.join(input_dir, f'subreddit_text_{split}.json')
        if os.path.exists(subreddit_file):
            print(f"\n[SUBREDDIT TEXT]")
            process_subreddit_text_incremental(
                input_path=subreddit_file,
                output_dir=split_output_dir,
                temp_dir=split_temp_dir,
                model=model,
                batch_size=batch_size,
                save_every_n_batches=save_every_n_batches
            )
        else:
            print(f"  ✗ Subreddit file not found: {subreddit_file}")

        # Clean up temp directory for this split
        if os.path.exists(split_temp_dir):
            shutil.rmtree(split_temp_dir)

    # Clean up dataset temp directory
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    print(f"\n✓ Dataset {dataset_name} complete!")
    print(f"  Output saved to: {output_dir}")

In [ ]:
# Process all datasets
total_start = time.time()

for dataset_name, input_dir in INPUT_DIRS.items():
    if os.path.exists(input_dir):
        process_dataset_incremental(
            dataset_name=dataset_name,
            input_dir=input_dir,
            output_base=OUTPUT_BASE_DIR,
            temp_base=TEMP_DIR,
            model=model,
            batch_size=BATCH_SIZE,
            save_every_n_batches=SAVE_EVERY_N_BATCHES
        )
    else:
        print(f"\n⚠ Skipping {dataset_name} - directory not found")

# Final cleanup of temp directory
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)

print(f"\n{'='*70}")
print(f"ALL PROCESSING COMPLETE!")
print(f"Total time: {time.time() - total_start:.1f}s")
print(f"{'='*70}")

## 7. Verify Output Structure

In [ ]:
# Verify the output structure
print("Output directory structure:")
print(f"\n{OUTPUT_BASE_DIR}/")

def print_tree(path, prefix="  "):
    if not os.path.exists(path):
        print(f"{prefix}(not found)")
        return

    items = sorted(os.listdir(path))
    for i, item in enumerate(items):
        item_path = os.path.join(path, item)
        is_last = (i == len(items) - 1)
        connector = "└── " if is_last else "├── "

        if os.path.isdir(item_path):
            print(f"{prefix}{connector}{item}/")
            new_prefix = prefix + ("    " if is_last else "│   ")
            print_tree(item_path, new_prefix)
        else:
            # Show file size for .npy files
            if item.endswith('.npy'):
                size_mb = os.path.getsize(item_path) / (1024 * 1024)
                print(f"{prefix}{connector}{item} ({size_mb:.1f} MB)")
            else:
                print(f"{prefix}{connector}{item}")

print_tree(OUTPUT_BASE_DIR)

In [ ]:
# Load and inspect sample embedding files
print("Verifying embedding files...\n")

for dataset_name in INPUT_DIRS.keys():
    print(f"Dataset: {dataset_name}")

    for split in ['train', 'dev', 'test']:
        base_path = os.path.join(OUTPUT_BASE_DIR, dataset_name, "filtered", split)

        if not os.path.exists(base_path):
            print(f"  {split}: (not found)")
            continue

        print(f"  {split}:")

        # Check user embeddings
        user_full = os.path.join(base_path, f'user_embeddings_{FULL_DIM}.npy')
        user_mrl = os.path.join(base_path, f'user_embeddings_{MRL_DIM}.npy')

        if os.path.exists(user_full):
            emb = np.load(user_full)
            print(f"    user_embeddings_{FULL_DIM}: {emb.shape}")
        if os.path.exists(user_mrl):
            emb = np.load(user_mrl)
            print(f"    user_embeddings_{MRL_DIM}: {emb.shape}")

        # Check subreddit embeddings
        sub_full = os.path.join(base_path, f'subreddit_embeddings_{FULL_DIM}.npy')
        sub_mrl = os.path.join(base_path, f'subreddit_embeddings_{MRL_DIM}.npy')

        if os.path.exists(sub_full):
            emb = np.load(sub_full)
            print(f"    subreddit_embeddings_{FULL_DIM}: {emb.shape}")
        if os.path.exists(sub_mrl):
            emb = np.load(sub_mrl)
            print(f"    subreddit_embeddings_{MRL_DIM}: {emb.shape}")

    print()

In [ ]:
# Validate embeddings are properly normalized
print("Validating embedding normalization...\n")

sample_path = os.path.join(OUTPUT_BASE_DIR, "v4", "filtered", "train", f"user_embeddings_{FULL_DIM}.npy")

if os.path.exists(sample_path):
    sample_emb = np.load(sample_path)
    norms = np.linalg.norm(sample_emb, axis=1)

    print(f"Sample: {sample_path}")
    print(f"  Shape: {sample_emb.shape}")
    print(f"  Dtype: {sample_emb.dtype}")
    print(f"  L2 norms - min: {norms.min():.6f}, max: {norms.max():.6f}, mean: {norms.mean():.6f}")
    print(f"  Value range: [{sample_emb.min():.4f}, {sample_emb.max():.4f}]")

    if np.allclose(norms, 1.0, atol=1e-5):
        print("  ✓ Embeddings are properly L2-normalized")
    else:
        print("  ⚠ Embeddings may not be properly normalized")
else:
    print(f"Sample file not found: {sample_path}")

In [ ]:
# Inspect NaN values in embeddings
print("Inspecting NaN values in embeddings...\n")

for dataset_name in INPUT_DIRS.keys():
    print(f"Dataset: {dataset_name}")
    print("=" * 50)

    for split in ['train', 'dev', 'test']:
        base_path = os.path.join(OUTPUT_BASE_DIR, dataset_name, "filtered", split)

        if not os.path.exists(base_path):
            continue

        print(f"\n  {split}:")

        for emb_type in ['user', 'subreddit']:
            for dim in [FULL_DIM, MRL_DIM]:
                emb_path = os.path.join(base_path, f'{emb_type}_embeddings_{dim}.npy')

                if not os.path.exists(emb_path):
                    continue

                emb = np.load(emb_path)

                # Count NaN statistics
                nan_mask = np.isnan(emb)
                nan_per_row = nan_mask.any(axis=1)  # Rows with at least one NaN
                all_nan_rows = nan_mask.all(axis=1)  # Rows that are entirely NaN

                total_rows = emb.shape[0]
                rows_with_nan = nan_per_row.sum()
                rows_all_nan = all_nan_rows.sum()
                total_nan_values = nan_mask.sum()

                if rows_with_nan > 0:
                    print(f"    {emb_type}_embeddings_{dim}:")
                    print(f"      Total rows: {total_rows}")
                    print(f"      Rows with any NaN: {rows_with_nan} ({100*rows_with_nan/total_rows:.2f}%)")
                    print(f"      Rows entirely NaN: {rows_all_nan} ({100*rows_all_nan/total_rows:.2f}%)")
                    print(f"      Total NaN values: {total_nan_values} / {emb.size} ({100*total_nan_values/emb.size:.2f}%)")

                    # Show indices of first few NaN rows
                    nan_indices = np.where(nan_per_row)[0]
                    print(f"      First 10 NaN row indices: {nan_indices[:10].tolist()}")
                else:
                    print(f"    {emb_type}_embeddings_{dim}: ✓ No NaN values")

    print()

## 8. Memory Cleanup

In [ ]:
# Optional: Clean up GPU memory when done
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared!")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")